In [ ]:
import torch
import clip
from PIL import Image
import logging
import time

# 设置日志记录
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 检查 GPU 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# 加载 CLIP 模型和预训练权重
model, preprocess = clip.load("ViT-B/32", device=device)

# 加载示例图像并处理
try:
    image = preprocess(Image.open("example.jpg")).unsqueeze(0).to(device)
except FileNotFoundError:
    logger.error("The image file 'example.jpg' was not found.")
    raise
except Exception as e:
    logger.error(f"An error occurred while opening the image: {e}")
    raise

# 示例文本
text = ["A photo of a cat", "A photo of a dog"]
text_tokens = clip.tokenize(text).to(device)

# 使用模型生成图像和文本的 Embedding
start_time = time.time()
with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text_tokens)
end_time = time.time()
logger.info(f"Embedding generation took {end_time - start_time:.2f} seconds.")

# 将 Embedding 转换为标准化的向量
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

# 打印图像和文本特征
eventual_similarity = torch.matmul(image_features, text_features.T)
logger.info("Embeddings generated successfully!")
logger.info(f"Similarity scores between image and text: {eventual_similarity.cpu().numpy()}")
